Tutorial - Squeeze Film Dampers
======================

This tutorial demonstrates how to use the `SqueezeFilmDamper` class from ROSS to compute the dynamic coefficients of squeeze film dampers (SFDs) using classical short-bearing theory.

Squeeze film dampers are passive damping devices widely used in rotating machinery to suppress vibrations and improve rotordynamic stability. They operate by generating a hydrodynamic pressure in the thin oil film between a non-rotating sleeve and the journal, which is driven by the orbital motion of the rotor.

The `SqueezeFilmDamper` class implements an **analytical short-bearing model** based on closed-form solutions to the Reynolds equation. The main outputs for each operating speed are:

- Direct damping coefficient `cxx` [N·s/m]
- Direct stiffness coefficient `kxx` [N/m] (due to oil film inertia/cavitation effects)
- Maximum pressure in the oil film `p_max` [Pa]
- Pressure angle `theta` [rad]

Three bearing geometries are supported:

| `geometry` string | Description |
|-------------------|-------------|
| `'groove'` | Central groove, open ends (no end seals) |
| `'end_seals'` | Closed ends (end seals), no groove |
| `'groove-end_seals'` | Central groove **and** end seals |

Cavitation behavior is controlled by the `cavitation` parameter (`True` or `False`).

## Section 1: Groove geometry (open ends)

The groove geometry assumes a central circumferential groove that feeds oil into the damper and two open ends through which oil can escape freely. This is the most common configuration in practice.

The example below uses the parameters from the class docstring — a damper operating at three speeds with `eccentricity_ratio = 0.5`:

In [ ]:
import plotly.io as pio

pio.renderers.default = "notebook"

from ross.bearings.squeeze_film_damper import SqueezeFilmDamper
from ross.units import Q_

sfd_groove = SqueezeFilmDamper(
    n=0,                                       # Node in the FE model of the rotor
    frequency=Q_([18600, 20000, 22000], "rpm"), # Operating speeds
    axial_length=Q_(0.9, "inches"),             # Damper length
    journal_radius=Q_(2.55, "inches"),          # Journal radius
    radial_clearance=Q_(0.003, "inches"),       # Radial clearance
    eccentricity_ratio=0.5,                    # Eccentricity ratio (ε = e/c)
    lubricant="ISOVG32",                       # Lubricant type
    geometry="groove",                         # Central groove, open ends
    cavitation=True,                           # Enable cavitation model
)

The `SqueezeFilmDamper` class computes all coefficients analytically during instantiation — no iterative solver is required. The calculation is nearly instantaneous even for a large number of operating speeds.

**Key design parameter:** The `eccentricity_ratio` ε = e/c relates the orbit radius *e* to the radial clearance *c*. For well-behaved operation, values below 0.4 are recommended; higher values increase the nonlinearity of the film forces.

### 1.1: Execution time

To display the total computation time, use:

In [ ]:
sfd_groove.show_execution_time()

Because the model is analytical, the execution time is typically on the order of microseconds, regardless of the number of frequencies.

### 1.2: Coefficients comparison

To display in table format the damping coefficient `cxx`, stiffness coefficient `kxx`, maximum pressure `p_max`, and pressure angle `theta` for all analyzed speeds, use:

In [ ]:
sfd_groove.show_coefficients_comparison()

The coefficients can also be accessed directly as arrays:

In [ ]:
import numpy as np

print("cxx [N·s/m]:", sfd_groove.cxx)
print("kxx [N/m]:  ", sfd_groove.kxx)
print("p_max [Pa]: ", sfd_groove.p_max)
print("theta [deg]:", np.degrees(sfd_groove.theta))

**Note on the stiffness coefficient:** When `cavitation=True`, the oil film is modeled as acting only over the positive-pressure region (π-film model). This eliminates the antisymmetric stiffness cross-coupling term (`kxx = 0`) for the groove geometry. If `cavitation=False`, the full 2π-film model is used, which may produce a non-zero stiffness.

### 1.3: Show results

To display a comprehensive summary table with geometry, operating conditions, and computed coefficients for each analyzed speed, use:

In [ ]:
sfd_groove.show_results()

The results table includes:

| Parameter | Description |
|-----------|-------------|
| Operating Speed | Rotational speed [RPM] |
| Geometry Type | `groove`, `end_seals`, or `groove-end_seals` |
| Cavitation | Cavitation model enabled (`True`/`False`) |
| Axial Length | Damper length [m] |
| Journal Radius | Journal radius [m] |
| Radial Clearance | Radial clearance [m] |
| Eccentricity Ratio | ε = e/c [dimensionless] |
| Lubricant Viscosity | Dynamic viscosity at reference temperature [Pa·s] |
| Damping Coefficient | `cxx` [N·s/m] |
| Stiffness Coefficient | `kxx` [N/m] |
| Pressure Angle | θ [° and rad] |
| Maximum Pressure | Peak pressure in the film [Pa] |

## Section 2: End-seal geometry

The end-seal geometry models a damper with sealed ends — oil cannot escape axially. This significantly increases the damping capacity compared to the open-end configuration, but also introduces a non-negligible oil film stiffness.

Note that for the `end_seals` geometry, the stiffness coefficient `kxx` is computed from the Sommerfeld-type analytical expression. With `cavitation=True`, the stiffness term is set to zero (π-film assumption); with `cavitation=False`, the full 2π-film gives `kxx = 0` but doubles the damping.

In [ ]:
from ross.bearings.squeeze_film_damper import SqueezeFilmDamper
from ross.units import Q_

sfd_end_seals = SqueezeFilmDamper(
    n=0,
    frequency=Q_([18600, 20000, 22000], "rpm"),
    axial_length=Q_(0.9, "inches"),
    journal_radius=Q_(2.55, "inches"),
    radial_clearance=Q_(0.003, "inches"),
    eccentricity_ratio=0.5,
    lubricant="ISOVG32",
    geometry="end_seals",    # Sealed ends, no groove
    cavitation=True,
)

sfd_end_seals.show_coefficients_comparison()

### 2.1: Full results

In [ ]:
sfd_end_seals.show_results()

## Section 3: Groove with end seals

The `groove-end_seals` geometry combines both features: a central groove for oil supply and sealed ends. This provides the highest load capacity and damping among the three configurations. The formulation is identical to the groove model, but the full film length (not half) is used.

In [ ]:
from ross.bearings.squeeze_film_damper import SqueezeFilmDamper
from ross.units import Q_

sfd_groove_seals = SqueezeFilmDamper(
    n=0,
    frequency=Q_([18600, 20000, 22000], "rpm"),
    axial_length=Q_(0.9, "inches"),
    journal_radius=Q_(2.55, "inches"),
    radial_clearance=Q_(0.003, "inches"),
    eccentricity_ratio=0.5,
    lubricant="ISOVG32",
    geometry="groove-end_seals",   # Central groove + sealed ends
    cavitation=True,
)

sfd_groove_seals.show_coefficients_comparison()

### 3.1: Comparing geometries

The three geometries produce different coefficient magnitudes for the same bearing dimensions. The comparison below shows the damping coefficient `cxx` at the first speed for each configuration:

In [ ]:
print(f"{'Geometry':<22} {'cxx [N·s/m]':>15} {'kxx [N/m]':>15}")
print("-" * 55)
for sfd, name in [
    (sfd_groove,       "groove"),
    (sfd_end_seals,    "end_seals"),
    (sfd_groove_seals, "groove-end_seals"),
]:
    print(f"{name:<22} {sfd.cxx[0]:>15.4e} {sfd.kxx[0]:>15.4e}")

## Section 4: Effect of cavitation

When `cavitation=False`, the oil film is assumed to act over the full 360° (2π-film model). This increases the damping for the groove geometry but sets the stiffness to zero regardless. The comparison below illustrates the effect:

In [ ]:
from ross.bearings.squeeze_film_damper import SqueezeFilmDamper
from ross.units import Q_

sfd_no_cavitation = SqueezeFilmDamper(
    n=0,
    frequency=Q_([18600, 20000, 22000], "rpm"),
    axial_length=Q_(0.9, "inches"),
    journal_radius=Q_(2.55, "inches"),
    radial_clearance=Q_(0.003, "inches"),
    eccentricity_ratio=0.5,
    lubricant="ISOVG32",
    geometry="groove",
    cavitation=False,    # Full 2π-film model
)

print(f"{'Model':<25} {'cxx [N·s/m]':>15} {'kxx [N/m]':>15}")
print("-" * 58)
print(f"{'groove (cavitation=True)':<25} {sfd_groove.cxx[0]:>15.4e} {sfd_groove.kxx[0]:>15.4e}")
print(f"{'groove (cavitation=False)':<25} {sfd_no_cavitation.cxx[0]:>15.4e} {sfd_no_cavitation.kxx[0]:>15.4e}")

## Section 5: Effect of eccentricity ratio

The eccentricity ratio ε = e/c is the most influential parameter on SFD behavior. As ε increases, the damping grows nonlinearly and the stiffness rises sharply. The analysis is repeated for a range of eccentricity ratios to illustrate this sensitivity:

In [ ]:
from ross.bearings.squeeze_film_damper import SqueezeFilmDamper
from ross.units import Q_
import numpy as np

eccentricity_ratios = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

print(f"{'ε':>6} {'cxx [N·s/m]':>15} {'kxx [N/m]':>15} {'p_max [Pa]':>15}")
print("-" * 55)

for eps in eccentricity_ratios:
    sfd = SqueezeFilmDamper(
        n=0,
        frequency=Q_([18600], "rpm"),
        axial_length=Q_(0.9, "inches"),
        journal_radius=Q_(2.55, "inches"),
        radial_clearance=Q_(0.003, "inches"),
        eccentricity_ratio=eps,
        lubricant="ISOVG32",
        geometry="groove-end_seals",
        cavitation=True,
    )
    print(f"{eps:>6.1f} {sfd.cxx[0]:>15.4e} {sfd.kxx[0]:>15.4e} {sfd.p_max[0]:>15.4e}")

## Section 6: Integrating with a rotor model

The `SqueezeFilmDamper` inherits from `BearingElement`, so it can be inserted directly into a `Rotor` assembly like any other bearing element. Its `cxx` and `kxx` coefficients are automatically incorporated into the global damping and stiffness matrices.

A typical SFD installation involves a support spring (squirrel-cage) in series with the damper. In this example a simple rotor is assembled with a radial bearing and an SFD sharing the same node:

In [ ]:
import ross as rs
from ross.bearings.squeeze_film_damper import SqueezeFilmDamper
from ross.units import Q_

# Build a simple rotor
steel = rs.Material(name="Steel", rho=7810, E=211e9, G_s=81.2e9)
shaft = [rs.ShaftElement(L=0.25, idl=0, odl=0.05, material=steel) for _ in range(4)]
disk = rs.DiskElement.from_geometry(n=2, material=steel, width=0.07, i_d=0.05, o_d=0.28)

# Stiff radial bearings at the shaft ends
brg0 = rs.BearingElement(n=0, kxx=1e8, cxx=0)
brg4 = rs.BearingElement(n=4, kxx=1e8, cxx=0)

# SFD at node 1 (in addition to the radial bearing stiffness)
sfd = SqueezeFilmDamper(
    n=1,
    frequency=Q_([18600, 20000, 22000], "rpm"),
    axial_length=Q_(0.9, "inches"),
    journal_radius=Q_(2.55, "inches"),
    radial_clearance=Q_(0.003, "inches"),
    eccentricity_ratio=0.3,
    lubricant="ISOVG32",
    geometry="groove-end_seals",
    cavitation=True,
)

rotor = rs.Rotor(
    shaft_elements=shaft,
    disk_elements=[disk],
    bearing_elements=[brg0, brg4, sfd],
)

# Run modal analysis
modal = rotor.run_modal(speed=0)
print("Natural frequencies [Hz]:", modal.wn[:4] / (2 * 3.14159))

## Section 7: Available lubricants

The `SqueezeFilmDamper` class accepts the same lubricant strings as the other fluid-film bearing classes in ROSS. The viscosity used in the coefficient formulas corresponds to the value at the first reference temperature of the selected lubricant.

| Lubricant string | ISO grade | Dynamic viscosity at reference temperature |
|-----------------|-----------|-------------------------------------------|
| `'ISOVG32'` | ISO VG 32 | See `lubricants_dict['ISOVG32']['liquid_viscosity1']` |
| `'ISOVG46'` | ISO VG 46 | See `lubricants_dict['ISOVG46']['liquid_viscosity1']` |
| `'ISOVG68'` | ISO VG 68 | See `lubricants_dict['ISOVG68']['liquid_viscosity1']` |

In [ ]:
from ross.bearings.lubricants import lubricants_dict

print(f"{'Lubricant':<12} {'viscosity1 [Pa·s]':>20} {'T1 [K]':>10}")
print("-" * 45)
for name in ["ISOVG32", "ISOVG46", "ISOVG68"]:
    props = lubricants_dict[name]
    print(f"{name:<12} {props['liquid_viscosity1']:>20.4e} {props['temperature1']:>10.2f}")

The effect of lubricant grade on the damping coefficient is shown below — heavier oils produce proportionally higher damping:

In [ ]:
from ross.bearings.squeeze_film_damper import SqueezeFilmDamper
from ross.units import Q_

print(f"{'Lubricant':<12} {'cxx [N·s/m]':>15} {'p_max [Pa]':>15}")
print("-" * 45)

for grade in ["ISOVG32", "ISOVG46", "ISOVG68"]:
    sfd = SqueezeFilmDamper(
        n=0,
        frequency=Q_([18600], "rpm"),
        axial_length=Q_(0.9, "inches"),
        journal_radius=Q_(2.55, "inches"),
        radial_clearance=Q_(0.003, "inches"),
        eccentricity_ratio=0.5,
        lubricant=grade,
        geometry="groove-end_seals",
        cavitation=True,
    )
    print(f"{grade:<12} {sfd.cxx[0]:>15.4e} {sfd.p_max[0]:>15.4e}")